# 🚀 Your First RAG System

Let's build a working RAG system step by step.
By the end of this notebook you'll have something real you can extend.

**What we'll build:** A Q&A system over a small knowledge base using real embeddings.

In [1]:
# Install the one library we need
# !pip install sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load a small, fast embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!")

✅ Model loaded!


## Step 1 — Add Your Documents

In [3]:
# Your knowledge base — replace these with your own documents!
documents = [
    "Python is a high-level programming language known for its readable syntax.",
    "Lists in Python are created with square brackets: my_list = [1, 2, 3]",
    "A for loop in Python iterates over items: for item in my_list: print(item)",
    "Functions in Python are defined with the def keyword.",
    "Python dictionaries store key-value pairs: {'name': 'Alice', 'age': 30}",
    "To read a file in Python: with open('file.txt') as f: content = f.read()",
]

# Turn every document into a vector (embedding)
doc_embeddings = model.encode(documents)

print(f"Indexed {len(documents)} documents")
print(f"Each document → vector of {doc_embeddings.shape[1]} numbers")

Indexed 6 documents
Each document → vector of 384 numbers


## Step 2 — Search

In [4]:
# Turn the question into a vector too, then find the closest documents

question = "How do I loop through a list?"

# Embed the question
question_embedding = model.encode(question)

# Compare question vector to every document vector
scores = cosine_similarity([question_embedding], doc_embeddings)[0]

# Get the 2 most similar documents
top_indices = np.argsort(scores)[::-1][:2]

print(f"Question: '{question}'\n")
print("Most relevant documents:")
for idx in top_indices:
    print(f"  [{scores[idx]:.2f}] {documents[idx]}")

Question: 'How do I loop through a list?'

Most relevant documents:
  [0.65] A for loop in Python iterates over items: for item in my_list: print(item)
  [0.44] Lists in Python are created with square brackets: my_list = [1, 2, 3]


## Step 3 — Generate

In [5]:
# Build the prompt with the retrieved context
context = "\n".join(documents[i] for i in top_indices)

prompt = f"""Answer using only the information below.

Context:
{context}

Question: {question}
Answer:"""

print(prompt)

Answer using only the information below.

Context:
A for loop in Python iterates over items: for item in my_list: print(item)
Lists in Python are created with square brackets: my_list = [1, 2, 3]

Question: How do I loop through a list?
Answer:


In [6]:
# Send to LLM (simulated — swap with real API call)
#
# Real call with Anthropic:
#   from anthropic import Anthropic
#   client = Anthropic()
#   response = client.messages.create(
#       model="claude-sonnet-4-6", max_tokens=200,
#       messages=[{"role": "user", "content": prompt}]
#   )
#   print(response.content[0].text)

answer = "Use a for loop: for item in my_list: print(item)"

print(f"Q: {question}")
print(f"A: {answer}")

Q: How do I loop through a list?
A: Use a for loop: for item in my_list: print(item)


## Try It With Different Questions

In [7]:
# Let's wrap everything into one simple function

def ask(question, top_k=2):
    # 1. Embed the question
    q_emb = model.encode(question)

    # 2. Find relevant docs
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    context = "\n".join(documents[i] for i in top_idx)

    # 3. Build prompt
    prompt = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"

    # 4. (Would call LLM here)
    print(f"Q: {question}")
    print(f"Context used:")
    for i in top_idx:
        print(f"  - {documents[i]}")
    print()


ask("What is a dictionary in Python?")
ask("How do I read a file?")
ask("How do I create a function?")

Q: What is a dictionary in Python?
Context used:
  - Python is a high-level programming language known for its readable syntax.
  - Functions in Python are defined with the def keyword.

Q: How do I read a file?
Context used:
  - To read a file in Python: with open('file.txt') as f: content = f.read()
  - Python is a high-level programming language known for its readable syntax.

Q: How do I create a function?
Context used:
  - Functions in Python are defined with the def keyword.
  - Lists in Python are created with square brackets: my_list = [1, 2, 3]



## What You Just Built

```
documents → embeddings  (stored once)
question  → embedding   (at query time)
           ↓
    cosine similarity search
           ↓
    top-K relevant docs
           ↓
    inject into prompt
           ↓
    LLM generates grounded answer
```

This is the core of every RAG system — from a simple chatbot to enterprise search.

The rest of this course teaches you how to make each step **better**.

---
**Continue to Section 1 — Embeddings** to learn how those vectors work.